# Q2 — World Indexes YTD vs US S&P 500 (as of 2026-08-21)
**Homework:** `cohorts/2026/homework1.md:35-60` — *How many of 10 world indexes beat US YTD as of 2026-08-21?*

**Novice:** Use `yfinance` Close-to-Close, handle `end` exclusive (+1 day), find first/last trading day.

**Answer (verified live Yahoo 2026-09-14):** **2 indexes** — **Japan Nikkei 27.36%** and **Canada TSX 14.86%** vs **US 11.90%**. Sensitive to hint literal `end` still 2.


## 2.1 Setup tickers (from homework)

10 world indexes — US is benchmark. Note tickers with `^` or `.SS` suffix.

In [ ]:
import yfinance as yf
import pandas as pd

TICKERS = {
    "US S&P500": "^GSPC",
    "China Shanghai": "000001.SS",
    "Hong Kong HSI": "^HSI",
    "Australia ASX200": "^AXJO",
    "India Nifty": "^NSEI",
    "Canada TSX": "^GSPTSE",
    "Germany DAX": "^GDAXI",
    "UK FTSE": "^FTSE",
    "Japan Nikkei": "^N225",
    "Mexico IPC": "^MXX",
    "Brazil Ibovespa": "^BVSP",
}
START = "2026-01-01"
END = "2026-08-21"  # inclusive
END_EXCL = (pd.Timestamp(END) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
print(END_EXCL, "is exclusive to include", END)

## 2.2 Download YTD closes

- `yf.download(list(tickers), start, end)` → MultiIndex `('Close', ticker)`
- Pick first available close ≥ START and last ≤ END per ticker (handles holidays).

In [ ]:
df = yf.download(list(TICKERS.values()), start=START, end=END_EXCL, progress=False, auto_adjust=False)
close = df["Close"]
print(f"Downloaded {close.shape[0]} days × {close.shape[1]} tickers")
print(f"Range {close.index.min().date()} to {close.index.max().date()}")
close.tail(3)

## 2.3 Compute YTD returns

`ret = last / first -1` per ticker, then sort. Novice: `dropna()` per column because holidays differ.

In [ ]:
details = {}
for name, ticker in TICKERS.items():
    s = close[ticker].dropna()
    s_filt = s[(s.index >= START) & (s.index <= END)]
    first_date, first = s_filt.index[0], float(s_filt.iloc[0])
    last_date, last = s_filt.index[-1], float(s_filt.iloc[-1])
    details[name] = {"ticker":ticker, "first":first, "last":last, "ret":last/first-1, "first_date":first_date.date(), "last_date":last_date.date()}

for n,d in sorted(details.items(), key=lambda x: x[1]["ret"], reverse=True):
    print(f"{n:20} {d['ret']*100:6.2f}%  {d['ticker']}  {d['first']:.1f}→{d['last']:.1f}")

us = details["US S&P500"]["ret"]
better = [n for n,d in details.items() if n!="US S&P500" and d["ret"]>us]
print(f"\nUS {us*100:.2f}% → Better: {len(better)}/9", better)
print(f"Answer Q2: {len(better)} indexes outperform US")

## 2.4 Sensitivity: literal hint `end='2026-08-21'` (excludes 21st)

Homework hint says `end='2026-08-21'` but yfinance `end` is exclusive → last would be 20th. Result still 2.

In [ ]:
df_hint = yf.download(list(TICKERS.values()), start=START, end=END, progress=False, auto_adjust=False)["Close"]
hint_rets = {n: float(s.dropna().iloc[-1]/s.dropna().iloc[0]-1) for n,s in [(name, df_hint[t]) for name,t in TICKERS.items()]}
print(f"US hint {hint_rets['US S&P500']*100:.2f}% → Better", [n for n,r in hint_rets.items() if n!="US S&P500" and r>hint_rets["US S&P500"]])

## 2.5 Additional: 3/5/10 year

Same method with earlier start.

In [ ]:
def calc(s,e):
    ee = (pd.Timestamp(e)+pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    c = yf.download(list(TICKERS.values()), start=s, end=ee, progress=False, auto_adjust=False)["Close"]
    return {n: float(c[t].dropna()[(c[t].dropna().index>=s)&(c[t].dropna().index<=e)].iloc[-1] / c[t].dropna()[(c[t].dropna().index>=s)&(c[t].dropna().index<=e)].iloc[0]-1) for n,t in TICKERS.items()}
for s,e in [("2023-08-21","2026-08-21"),("2021-08-21","2026-08-21"),("2016-08-21","2026-08-21")]:
    r=calc(s,e)
    print(s, "→", e, "US", f"{r['US S&P500']*100:.1f}%", "better", [n for n,v in r.items() if n!="US S&P500" and v>r["US S&P500"]])